# arena-smoke

这份 notebook 只回答一个问题 —— **`arena-pyspark` kernel 真能起 Spark 吗？**

界面说「kernel 就绪」而这里跑不通，那句「就绪」就是假话，所以每次交付前都会把它从头执行一遍，
拿这一页的执行结果当最小正面证据。它**不带 outputs** —— 执行产物属于执行的那一刻，
把结果抄进 git 只会让「教程里的数字」变得不可追（也永远不知道是哪天、哪个镜像跑出来的）。

跑法：右上角选 `PySpark (arena)`，然后 Run All。
预期看到三行：`python /opt/arena-ide-env/...`、`rows 15`、`venv ok`。
那个 15 是 `range(6)` 里 id 的**和**（0+1+2+3+4+5 = 15），不是行数 6 ——
数字对不上就说明跑它的不是那个 kernel，请去查 kernelspec，别去改这里期望的数字。

In [ ]:
import sys

print("python", sys.executable)

In [ ]:
from pyspark.sql import SparkSession, functions as f

spark = SparkSession.builder.appName("arena-smoke").getOrCreate()
# 打的是 id 之和 0+1+2+3+4+5 = 15（range(6).count() 只会给 6，那是行数）。
# 用聚合而不只数行数：它顺带证明 Spark 真做了一次计算，而不只是把 session 对象建了出来。
# 外面那层 int() 是把 bigint 列的返回值收成 Python 整数，打印形状必须是 "rows 15"。
print("rows", int(spark.range(6).agg(f.sum("id")).collect()[0][0]))

In [ ]:
# 解释器必须在 IDE 的 venv 里 —— 这是那条隔离红线的正面证据。
# 落回系统 python 的那一刻起，notebook 里装的包就进了判题子进程能看到的那套 site-packages，
# 而 docker/BUILDINFO.md 承诺的是「重建镜像即可复现」。这条 assert 红了别去改它，去查 kernelspec。
assert sys.executable.startswith("/opt/arena-ide-env/"), sys.executable
print("venv ok")